# Les polynômes sont des vecteurs

| Module | Durée | Prérequis |
|---|---|---|
| M5 – Espaces vectoriels et bases | 75 min | M4 |

In [ ]:
# Style graphique Liouaï : couleurs du logo, axes sobres
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

LIOUAI = {"navy": "#243743", "vert": "#28B78D", "vert_texte": "#167A5C", "gris": "#A1A9AE",
          "gris_texte": "#5C6870", "jaune": "#E5C800", "fond": "#F3F5F6"}
plt.rcParams.update({
    "axes.prop_cycle": plt.cycler(color=[LIOUAI["vert"], LIOUAI["navy"], LIOUAI["gris"],
                                         LIOUAI["jaune"], LIOUAI["vert_texte"], LIOUAI["gris_texte"]]),
    "axes.edgecolor": LIOUAI["gris"], "axes.labelcolor": LIOUAI["navy"],
    "axes.titlecolor": LIOUAI["navy"], "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": LIOUAI["gris_texte"], "ytick.color": LIOUAI["gris_texte"],
    "text.color": LIOUAI["navy"], "grid.color": "#E1E6E9", "legend.frameon": False,
    "font.family": "sans-serif", "font.sans-serif": ["Roboto", "Helvetica", "Arial", "DejaVu Sans"],
    "figure.dpi": 100,
})
# Dégradé de marque : gris clair -> vert -> navy (et sa version inversée)
if "liouai" not in mpl.colormaps:
    _cmap = LinearSegmentedColormap.from_list("liouai", [LIOUAI["fond"], LIOUAI["vert"], LIOUAI["navy"]])
    mpl.colormaps.register(_cmap)
    mpl.colormaps.register(_cmap.reversed())

## ① Définir la question

La station météo du collège enregistre la température toutes les trois heures, de 6 h à 21 h. Pour un exposé sur les variations de température au cours d'une journée, Camille veut tracer une **courbe lisse** qui passe exactement par les six relevés, et estimer la température à 14 h, l'heure de la récréation.

Six points, une courbe lisse : on pense à un **polynôme** de degré 5,

$$
p(t) = a_0 + a_1 t + a_2 t^2 + a_3 t^3 + a_4 t^4 + a_5 t^5 .
$$

Camille trouve les coefficients avec un tableur. Son camarade Hugo refait le calcul avec des relevés arrondis différemment, au dixième de degré près : ses coefficients n'ont **plus rien à voir** avec ceux de Camille. Pourtant, leurs deux courbes se superposent presque parfaitement.

**Les questions : comment des coefficients si différents peuvent-ils décrire presque la même courbe ? Existe-t-il une meilleure manière de décrire ce polynôme ?**

**Activité** : Les coefficients $a_0, \dots, a_5$ ont-ils un sens physique pour Camille ? Lequel l'intéresse vraiment : les coefficients, ou la courbe ?

Indiquez vos réponses et justifiez-les.

## ② Estimer avant de calculer

Les relevés sont 7,8 °C à 6 h, 10,9 °C à 9 h, 15,6 °C à 12 h, 18,4 °C à 15 h, 16,9 °C à 18 h et 12,1 °C à 21 h.

In [ ]:
pari_temperature_14h = 17.0          # en °C (modifiez)
pari_effet_erreur_005 = "1 %"        # variation des coefficients pour une erreur de 0,05 °C : "1 %", "10 %" ou "50 %" (modifiez)

## ③ Abstraire : les polynômes forment un espace vectoriel

On peut additionner deux polynômes de degré au plus 5, et les multiplier par un nombre : on obtient encore un polynôme de degré au plus 5. Ces opérations ont les mêmes propriétés que pour les vecteurs de $\mathbb{R}^6$. Les polynômes de degré au plus 5 forment donc un **espace vectoriel**, de dimension 6.

Un polynôme est un vecteur, et ses coordonnées dépendent de la base choisie.

- Dans la **base canonique** $1, t, t^2, \dots, t^5$, les coordonnées sont les coefficients $a_0, \dots, a_5$. Pour les trouver, on écrit $p(t_i) = y_i$ pour chaque relevé : c'est un système $V\mathbf{a} = \mathbf{y}$, où $V$ est la **matrice de Vandermonde**, de ligne $i$ égale à $(1, t_i, t_i^2, \dots, t_i^5)$.
- Dans la **base de Lagrange** $L_0, \dots, L_5$, chaque $L_j$ vaut 1 à l'heure $t_j$ et 0 aux cinq autres heures :

$$
L_j(t) = \prod_{k \neq j} \frac{t - t_k}{t_j - t_k}, \qquad p = y_0 L_0 + y_1 L_1 + \dots + y_5 L_5 .
$$

Dans cette base, les coordonnées sont **les relevés eux-mêmes** : il n'y a aucun système à résoudre.

![Schéma : le même polynôme a pour coordonnées les coefficients a dans la base canonique et les valeurs y dans la base de Lagrange ; on passe de l'un à l'autre par la matrice de Vandermonde](../../assets/schemas/schema-coordonnees-polynome.svg)

**Activité** : Avec trois points $(0\,;\,1)$, $(1\,;\,3)$, $(2\,;\,2)$ et un polynôme de degré 2 :

1. écrivez le système $V\mathbf{a} = \mathbf{y}$ et résolvez-le ;
2. écrivez $L_0$, $L_1$, $L_2$, puis développez $1 \cdot L_0 + 3 \cdot L_1 + 2 \cdot L_2$. Retrouvez-vous le même polynôme ?

Indiquez vos réponses et justifiez-les.

## ④ Calculer

### Dans la base canonique

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

t = np.array([6, 9, 12, 15, 18, 21], dtype=float)
y = np.array([7.8, 10.9, 15.6, 18.4, 16.9, 12.1])

V = np.vander(t, increasing=True)            # colonnes : 1, t, t², ..., t⁵
a = np.linalg.solve(V, y)
print("Coefficients a0 ... a5 :", a)

def p_canonique(coef, x):
    return np.vander(np.atleast_1d(x), len(coef), increasing=True) @ coef

print(f"Température estimée à 14 h : {p_canonique(a, 14.0)[0]:.2f} °C")

### Dans la base de Lagrange

In [ ]:
def base_lagrange(noeuds, x):
    # Renvoie une matrice : ligne = point x, colonne j = valeur de L_j en ce point
    x = np.atleast_1d(x)
    L = np.ones((len(x), len(noeuds)))
    for j, tj in enumerate(noeuds):
        for k, tk in enumerate(noeuds):
            if k != j:
                L[:, j] *= (x - tk) / (tj - tk)
    return L

def p_lagrange(noeuds, valeurs, x):
    return base_lagrange(noeuds, x) @ valeurs

print(f"Température estimée à 14 h : {p_lagrange(t, y, 14.0)[0]:.2f} °C")
x = np.linspace(6, 21, 301)
print("Écart maximal entre les deux courbes :", np.abs(p_canonique(a, x) - p_lagrange(t, y, x)).max())

### Voir les deux bases

On trace les vecteurs de chaque base sur la journée. Pour comparer les fonctions $t^k$, qui prennent des valeurs très différentes, on les divise par leur valeur à 21 h.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for k in range(6):
    axes[0].plot(x, x ** k / 21 ** k, label=f"$t^{k}$")
axes[0].set_title("Base canonique (mise à l'échelle)"); axes[0].legend(fontsize=8)

L = base_lagrange(t, x)
for j in range(6):
    axes[1].plot(x, L[:, j], label=f"$L_{j}$")
axes[1].axhline(0, color=LIOUAI["gris"], lw=0.8); axes[1].plot(t, np.ones(6), "o", color=LIOUAI["navy"], ms=4)
axes[1].set_title("Base de Lagrange"); axes[1].legend(fontsize=8, ncol=2)

axes[2].plot(x, p_lagrange(t, y, x), color=LIOUAI["vert"], lw=2, label="polynôme interpolateur")
axes[2].plot(t, y, "o", color=LIOUAI["navy"], label="relevés")
axes[2].plot(14, p_lagrange(t, y, 14.0)[0], "s", color=LIOUAI["jaune"], ms=8, label="estimation à 14 h")
axes[2].set_title("La courbe de Camille"); axes[2].set_ylabel("°C"); axes[2].legend(fontsize=8)
for ax in axes:
    ax.set_xlabel("heure"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

**Activité** : Les courbes $t^4$ et $t^5$ se ressemblent beaucoup sur la journée. Au module M1, on mesurait la ressemblance de deux vecteurs par le cosinus de leur angle. Que vaut-il ici, selon vous ?

In [ ]:
def cosinus(u, v):
    return u @ v / (np.linalg.norm(u) * np.linalg.norm(v))

print(f"cos(t⁴, t⁵) sur la journée : {cosinus(x ** 4, x ** 5):.4f}")
print(f"conditionnement de V        : {np.linalg.cond(V):.2e}")

Des vecteurs de base presque parallèles forment une base **presque dépendante** : la matrice $V$ est presque singulière, et son conditionnement est énorme (module M4).

### L'expérience d'Hugo : une petite erreur sur les relevés

In [ ]:
rng = np.random.default_rng(1)
y_hugo = y + rng.normal(0, 0.05, 6)          # relevés arrondis autrement : écarts de l'ordre de 0,05 °C
a_hugo = np.linalg.solve(V, y_hugo)

print(f"{'':4s}{'Camille':>12s}{'Hugo':>12s}{'variation':>11s}")
for k in range(6):
    print(f"a{k}  {a[k]:12.3e}{a_hugo[k]:12.3e}{(a_hugo[k] - a[k]) / abs(a[k]):11.0%}")
print(f"\nVariation relative des relevés     : {np.linalg.norm(y_hugo - y) / np.linalg.norm(y):.2%}")
print(f"Variation relative des coefficients : {np.linalg.norm(a_hugo - a) / np.linalg.norm(a):.0%}")
print(f"Écart maximal entre les deux courbes : {np.abs(p_canonique(a_hugo, x) - p_canonique(a, x)).max():.2f} °C")

### Une meilleure base canonique : changer d'origine et d'unité

Le problème vient en partie des grandes valeurs de $t$ : $21^5 \approx 4 \times 10^6$. On change de variable avec $s = (t - 13{,}5) / 7{,}5$, qui va de $-1$ à $1$ sur la journée. C'est encore un changement de base dans l'espace des polynômes.

In [ ]:
s = (t - 13.5) / 7.5
Vs = np.vander(s, increasing=True)
print(f"conditionnement avec t : {np.linalg.cond(V):.2e}")
print(f"conditionnement avec s : {np.linalg.cond(Vs):.1f}")
b, b_hugo = np.linalg.solve(Vs, y), np.linalg.solve(Vs, y_hugo)
print(f"Variation relative des coefficients en s : {np.linalg.norm(b_hugo - b) / np.linalg.norm(b):.2%}")

### Une mise en garde : le phénomène de Runge

Interpoler par un polynôme de haut degré n'est pas toujours une bonne idée. Prenons la fonction $f(x) = 1 / (1 + 25x^2)$ et des points équidistants de plus en plus nombreux.

In [ ]:
f = lambda u: 1 / (1 + 25 * u ** 2)
u = np.linspace(-1, 1, 401)
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.plot(u, f(u), color=LIOUAI["navy"], lw=2, label="fonction f")
for n_points, couleur in [(7, LIOUAI["vert"]), (11, LIOUAI["jaune"]), (15, LIOUAI["gris"])]:
    noeuds = np.linspace(-1, 1, n_points)
    ax.plot(u, p_lagrange(noeuds, f(noeuds), u), color=couleur, label=f"{n_points} points")
ax.set_ylim(-1.4, 2.1); ax.legend(loc="lower center", ncol=4); ax.grid(alpha=0.3)
ax.set_title("Plus de points ne veut pas dire meilleure courbe")
plt.tight_layout(); plt.show()

## ⑤ Interpréter

In [ ]:
print(f"Votre pari à 14 h : {pari_temperature_14h} °C ; estimation : {p_lagrange(t, y, 14.0)[0]:.1f} °C")
print(f"Votre pari sur l'effet d'une erreur de 0,05 °C : {pari_effet_erreur_005} ; "
      f"mesuré : {np.linalg.norm(a_hugo - a) / np.linalg.norm(a):.0%}")

**Activité** :

1. Expliquez à Camille et Hugo, en quelques phrases, pourquoi leurs coefficients diffèrent tant alors que leurs courbes se ressemblent.
2. Dans quelle base les coordonnées du polynôme sont-elles les plus **stables** ? les plus **faciles à interpréter** ?
3. L'estimation à 14 h est-elle fiable ? Et une estimation à 3 h du matin la nuit suivante (soit $t = 27$) avec le même polynôme ? Essayez.
4. Que montre le phénomène de Runge sur l'idée de « faire passer une courbe par tous les points » ?

Indiquez vos réponses et justifiez-les.

## ⑥ Critiquer et généraliser

- **Hypothèse fragile.** On a supposé que les relevés sont exacts. Avec des mesures bruitées, faut-il vraiment une courbe qui passe **par** tous les points ? Le module M6 proposera une autre approche : la courbe qui passe **au plus près**.
- **Vérifier autrement.** Les colonnes de $V^{-1}$ sont les coordonnées canoniques des polynômes $L_j$ : c'est la matrice de passage de la base de Lagrange vers la base canonique. Vérifiez-le pour $L_2$.
- **Généraliser.** Écrivez une fonction `conditionnement(n)` qui renvoie le conditionnement de la matrice de Vandermonde pour $n$ relevés équidistants entre 6 h et 21 h, en variable $t$ puis en variable $s$. Tracez les deux courbes pour $n$ de 3 à 15, en échelle logarithmique.

In [ ]:
def conditionnement(n):
    # À compléter : renvoie (conditionnement en t, conditionnement en s)
    ...

## Corrigé

*À n'ouvrir qu'après avoir cherché.*

In [ ]:
def conditionnement(n):
    tn = np.linspace(6, 21, n)
    sn = (tn - 13.5) / 7.5
    return np.linalg.cond(np.vander(tn, increasing=True)), np.linalg.cond(np.vander(sn, increasing=True))

ns = range(3, 16)
c = np.array([conditionnement(n) for n in ns])
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.semilogy(ns, c[:, 0], "o-", label="variable t (heures)")
ax.semilogy(ns, c[:, 1], "o-", label="variable s (entre −1 et 1)")
ax.set_xlabel("nombre de relevés"); ax.set_ylabel("conditionnement"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

# Colonnes de V^-1 = coefficients des polynômes de Lagrange
coef_L2 = np.linalg.inv(V)[:, 2]
print("L2 calculé par V^-1 et par la formule coïncident ?",
      np.allclose(p_canonique(coef_L2, x), base_lagrange(t, x)[:, 2]))
for h in [24, 27, 30]:
    print(f"Extrapolation à t = {h} h : {p_lagrange(t, y, float(h))[0]:5.1f} °C")
print("Hors de la plage des relevés, le polynôme s'emballe : l'extrapolation n'a aucun sens physique.")

## Bilan — auto-évaluation

Pour chaque compétence travaillée dans ce notebook, choisissez un niveau : **0** non observé · **1** guidé · **2** autonome · **3** transfert.
La cellule enregistre votre bilan dans `progression.json`, à la racine du site.

In [ ]:
import json, datetime, pathlib

bilan = {
    "Repérer ressemblances et différences entre situations pour abstraire": 1,
    "Comparer les mérites des outils dans le contexte": 1,
    "Avoir l'intuition des mérites relatifs du concept": 1,
    "Lister les sources d'erreur liées aux limites du calcul": 1,
    "Sentir si le résultat est plausible pour le contexte": 1,
}   # modifiez les niveaux

ici = pathlib.Path.cwd()
racine = next((p for p in [ici, *ici.parents] if (p / "myst.yml").exists()), ici)
fichier = racine / "progression.json"
historique = json.loads(fichier.read_text(encoding="utf-8")) if fichier.exists() else []
historique.append({"notebook": "5.3-polynomes-vecteurs", "date": datetime.date.today().isoformat(), "niveaux": bilan})
fichier.write_text(json.dumps(historique, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Bilan enregistré ({len(historique)} entrée(s) dans {fichier.name}).")